# Destiny Explained Studio 0.8.1 — Blender Bridge

Notebook préparé pour Colab Pro, également utilisable sur CPU. Il produit **au plus trois PNG**, jamais une vidéo. GPU non garanti : Blender vérifie lui-même les backends et se replie sur CPU. Aucun service de génération, clé API ou asset propriétaire.

Avant fusion humaine : choisir REF `feat/blender-bridge-0.8.1`. Après fusion : `main`. Le notebook existant Remotion/Kokoro reste inchangé. Installer les dépendances nécessite Internet ; les snapshots et images se calculent ensuite localement. Les scènes sont des reconstitutions artistiques.


In [ ]:
# 1 — Installer/détecter les outils (exécuter une fois)
import os, sys, json, shutil, subprocess, platform, urllib.request, hashlib, tarfile
from pathlib import Path
assert sys.version_info >= (3, 11), "Utiliser Python 3.11 ou plus récent."
REF = "main" # @param ["main", "feat/blender-bridge-0.8.1"]
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch','https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
else:
    if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Le checkout contient des modifications : les sauvegarder avant de changer REF.')
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin',f'refs/heads/{REF}:refs/remotes/origin/{REF}'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'switch',REF],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only',f'origin/{REF}'],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()!=subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',f'origin/{REF}'],text=True).strip():
        raise RuntimeError('Commits locaux conservés : utiliser une session propre pour cette REF.')
ROOT = PROJECT/'remotion'
if not (ROOT/'blender/import_scene.py').is_file():
    raise RuntimeError('Cette REF ne contient pas le bridge. Pendant la revue, choisir feat/blender-bridge-0.8.1.')
# Node officiel et version verrouillée ; vérifier le SHA publié avant extraction.
version='22.23.4'
existing=shutil.which('node')
if not existing or subprocess.check_output([existing,'-p','process.versions.node'],text=True).strip()!=version:
    arch={'x86_64':'x64','aarch64':'arm64'}.get(platform.machine())
    if not arch: raise RuntimeError('Architecture Node non prise en charge par ce notebook.')
    name=f'node-v{version}-linux-{arch}.tar.xz';base=f'https://nodejs.org/dist/v{version}/'
    cache=Path('/content/destiny-tools');cache.mkdir(exist_ok=True)
    archive=cache/name
    urllib.request.urlretrieve(base+name,archive)
    manifest=urllib.request.urlopen(base+'SHASUMS256.txt').read().decode()
    expected=next(line.split()[0] for line in manifest.splitlines() if line.split()[-1]==name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest()==expected, 'SHA Node incorrect'
    with tarfile.open(archive) as bundle: bundle.extractall(cache,filter='data')
    os.environ['PATH']=str(cache/f'node-v{version}-linux-{arch}'/'bin')+os.pathsep+os.environ['PATH']
NODE=shutil.which('node')
subprocess.run(['npm','ci','--no-audit','--no-fund'],cwd=ROOT,check=True)
BLENDER=shutil.which('blender')
if not BLENDER:
    if os.geteuid()!=0: raise RuntimeError('Installer Blender via le gestionnaire système, puis relancer cette cellule.')
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','blender'],check=True)
    BLENDER=shutil.which('blender')
if not BLENDER: raise RuntimeError('Blender non installé.')
print(subprocess.check_output([BLENDER,'--version'],text=True).splitlines()[0])
print('Commit:',subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# 2 — Ressources observées (la sélection effective sera dans chaque rapport Blender)
cpus=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count()
print('CPU disponibles:',cpus)
print(next((line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')),'RAM inconnue'))
if shutil.which('nvidia-smi'):
    gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
    print(gpu.stdout.strip() if gpu.returncode==0 else 'GPU non accessible : repli CPU possible.')
else: print('Pas de nvidia-smi : Blender vérifiera ses backends, puis choisira CPU si nécessaire.')


## Entrée et limites

Choisir l’exemple ou téléverser **un seul JSON** de projet Studio (2 Mio maximum) ou snapshot bridge (4 Mio). Le projet reste intact ; le résultat est écrit sous `remotion/out/blender-colab`. Les projets avec personnages historiques sans rig actif, ou les frames de cartes/texte, sont refusés explicitement.

Les snapshots figent les poses 0.7 ; ils n’importent ni le montage, ni les captions, ni l’audio dans Blender. Le décor est un sous-ensemble procédural signalé dans `omissions`. Les images seront utilisées pour le montage Remotion dans une prochaine itération ; cette intégration n’est pas automatique ici.


In [ ]:
# 3 — Préparer au plus trois snapshots
USE_EXAMPLE = True # @param {type:"boolean"}
FRAMES = "210,240,660" # @param {type:"string"}
PROFILE = "standard" # @param ["standard", "cinematic"]
SCALE = 25 # @param {type:"slider", min:10, max:100, step:5}
DEVICE = "auto" # @param ["auto", "cpu"]
OUT=ROOT/'out/blender-colab';OUT.mkdir(parents=True,exist_ok=True)
if USE_EXAMPLE:
    data=json.loads((ROOT/'episodes/hybrid_riis_07.json').read_text())
    data['shorts']['polish']['detailBudget']=6 # copie explicite, pas modification du projet de référence
else:
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('Sélectionner exactement un JSON, sans ZIP.')
    payload=next(iter(uploaded.values()))
    if len(payload)>4*1024*1024:raise ValueError('Entrée supérieure à 4 Mio.')
    data=json.loads(payload.decode('utf-8'))
if not isinstance(data,dict):raise ValueError('Un objet JSON de projet ou snapshot est nécessaire.')
source=OUT/'input.json';source.write_text(json.dumps(data,ensure_ascii=False))
if data.get('format')=='destiny-blender-scene':
    # Un snapshot est déjà figé : ne pas simuler d’autres frames en changeant son numéro.
    snapshots=[source];print('Snapshot figé : une seule image sera rendue.')
else:
    if source.stat().st_size>2*1024*1024:raise ValueError('Projet Studio supérieur à 2 Mio.')
    requested=[int(x.strip()) for x in FRAMES.split(',') if x.strip()]
    if not 1<=len(requested)<=3 or len(set(requested))!=len(requested):raise ValueError('Choisir 1 à 3 frames distinctes, sans troncature automatique.')
    snapshots=[]
    for frame in requested:
        snapshot=OUT/f'frame-{frame:05d}.json'
        subprocess.run([NODE,'--experimental-strip-types','blender/export.mjs',str(source),str(frame),str(snapshot),PROFILE],cwd=ROOT,check=True)
        snapshots.append(snapshot)
print('Snapshots:',[x.name for x in snapshots])


In [ ]:
# 4 — Rendre seulement les PNG demandés ; aucun rendu vidéo
import time
results=[]
for snapshot in snapshots:
    frame=json.loads(snapshot.read_text())['source']['frame']
    output=OUT/f'frame-{frame:05d}-{PROFILE}.png'
    command=[BLENDER,'--background','--factory-startup','--python-exit-code','1','--python',str(ROOT/'blender/import_scene.py'),'--',str(snapshot),str(output),'--profile',PROFILE,'--device',DEVICE,'--scale',str(SCALE),'--threads',str(max(1,min(cpus or 1,4)))]
    started=time.monotonic();log=output.with_suffix('.log')
    with log.open('w') as stream:run=subprocess.run(command,stdout=stream,stderr=subprocess.STDOUT)
    if run.returncode:
        print(log.read_text()[-8000:]);raise RuntimeError('Rendu arrêté ; examiner le journal. Ne pas relancer en boucle.')
    report=json.loads(output.with_suffix('.report.json').read_text())
    if not output.is_file() or not report['rendered']:raise RuntimeError('Aucun PNG produit.')
    print(frame,round(time.monotonic()-started,2),'s',output.stat().st_size,'octets',report['configuration'])
    results.append(output)


In [ ]:
# 5 — Prévisualiser et récupérer images, snapshots, logs et rapports
from IPython.display import display, Image
from google.colab import files
for output in results:display(Image(filename=str(output),width=270))
archive=shutil.make_archive(str(ROOT/'out/blender-bridge-results'),'zip',OUT)
files.download(archive)


## Reprise et validation

Télécharger le ZIP avant la fin de session. Corriger une frame non voxel dans le formulaire, sans modifier les faits du script. Si un GPU détecté échoue au rendu, le bridge tente une fois le CPU et le note ; un échec CPU arrête la cellule. Le débruitage cinematic est désactivé explicitement si le Blender installé ne l’annonce pas.

Notebook préparé et vérifié structurellement ; son exécution dans un **Colab Pro distant n’est pas déclarée validée**. Blender 4.3.2 est validé localement sur CPU ; les versions fournies par APT et les GPU Colab restent à tester. Aucune image ni vidéo lourde ne doit être ajoutée à Git. Voir `docs/BLENDER-BRIDGE-PROGRESS.md` pour l’état et les commandes de reprise.
